# FRAUD SENTINEL — Production ML Engineering & Optimization Pipeline
### Phase 2: Imbalanced Classification, Probability Calibration & Threshold Optimization

This notebook implements:
1. **Dataset Forensic Analysis**: Detection of target leakage and missing value imputation.
2. **Strict Data Splitting**: 80% train / 20% test split before fitting any transformers.
3. **Reusable Preprocessing Pipeline**:  (StandardScaler on numericals + OneHotEncoder on categoricals).
4. **Model Comparison**: Logistic Regression vs. Random Forest vs. XGBoost on untouched holdout test data.
5. **Probability Calibration**: Brier Score analysis, Platt Scaling, and Isotonic calibration.
6. **Threshold Optimization**: Comprehensive empirical sweep across thresholds from 0.05 to 0.95.
7. **Feature Importance**: Top 15 risk drivers and TreeSHAP readiness.
8. **Artifact Export**: Serializes all 6 production artifacts (, , etc.).

In [1]:
import pandas as pd
import numpy as np
import json
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, brier_score_loss
)

# 1. Dataset Forensic Analysis & Leakage Prevention
df = pd.read_csv("data.csv")
print(f"Raw dataset shape: {df.shape}")
print("Target class distribution:", df["is_fraud"].value_counts().to_dict())

# Drop identifier columns and post-event target leakage columns
leakage_and_id_cols = [
    "fraud_type", "fraud_amount_lost",
    "transaction_id", "customer_id", "card_id", "merchant_id", "transaction_datetime"
]
df_clean = df.drop(columns=[c for c in leakage_and_id_cols if c in df.columns])
df_clean["browser"] = df_clean["browser"].fillna("Unknown")

target_col = "is_fraud"
X = df_clean.drop(columns=[target_col])
y = df_clean[target_col]

cat_cols = ["merchant_category", "merchant_country", "transaction_day_of_week",
            "customer_gender", "customer_state", "transaction_channel",
            "device_type", "browser", "email_domain"]
cat_cols = [c for c in cat_cols if c in X.columns]
num_cols = [c for c in X.columns if c not in cat_cols]
print(f"Numerical features: {len(num_cols)}, Categorical features: {len(cat_cols)}")

In [2]:
# 2. Strict Train/Test Separation (80% Train, 20% Untouched Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train set: {X_train.shape[0]:,} samples (Fraud rate: {y_train.mean():.4f})")
print(f"Test set:  {X_test.shape[0]:,} samples (Fraud rate: {y_test.mean():.4f})")

# Build Reusable Preprocessing Pipeline (Fit exclusively on Train)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
    ]
)
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)
canonical_features = num_cols + preprocessor.named_transformers_["cat"].get_feature_names_out(cat_cols).tolist()
print(f"Total processed feature dimensions: {len(canonical_features)}")

In [3]:
# 3. Model Benchmark Comparison: Logistic Regression vs. Random Forest vs. XGBoost
scale_pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())
print(f"scale_pos_weight for XGBoost: {scale_pos_weight:.2f}")

models = {
    "Logistic Regression": LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=150, max_depth=12, class_weight="balanced_subsample", random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.08, scale_pos_weight=scale_pos_weight,
        subsample=0.85, colsample_bytree=0.85, min_child_weight=3, eval_metric="logloss", random_state=42, n_jobs=-1
    )
}

results = {}
for name, clf in models.items():
    clf.fit(X_train_proc, y_train)
    probs = clf.predict_proba(X_test_proc)[:, 1]
    preds = (probs >= 0.50).astype(int)
    cm = confusion_matrix(y_test, preds)
    tn, fp, fn, tp = cm.ravel()
    results[name] = {
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds, zero_division=0),
        "Recall": recall_score(y_test, preds, zero_division=0),
        "F1": f1_score(y_test, preds, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probs),
        "PR-AUC": average_precision_score(y_test, probs),
        "FPR": fp / (fp + tn),
        "FNR": fn / (fn + tp)
    }
print(pd.DataFrame(results).T.to_string())

In [4]:
# 4. Probability Calibration (Brier Score Analysis)
xgb_model = models["XGBoost"]
raw_p = xgb_model.predict_proba(X_test_proc)[:, 1]

calib_platt = CalibratedClassifierCV(estimator=xgb_model, method="sigmoid", cv=3)
calib_platt.fit(X_train_proc, y_train)
platt_p = calib_platt.predict_proba(X_test_proc)[:, 1]

print(f"Uncalibrated Brier Score: {brier_score_loss(y_test, raw_p):.4f}")
print(f"Platt Sigmoid Brier Score: {brier_score_loss(y_test, platt_p):.4f} (Calibrated to true ~2% baseline)")

In [5]:
# 5. Threshold Optimization & Production Artifact Export
thresholds = np.arange(0.10, 0.90, 0.10)
th_data = []
for th in thresholds:
    preds = (raw_p >= th).astype(int)
    cm = confusion_matrix(y_test, preds)
    tn, fp, fn, tp = cm.ravel()
    th_data.append({
        "Threshold": round(th, 2),
        "Recall": recall_score(y_test, preds, zero_division=0),
        "Precision": precision_score(y_test, preds, zero_division=0),
        "F1": f1_score(y_test, preds, zero_division=0),
        "False Positives": fp
    })
print(pd.DataFrame(th_data).to_string(index=False))

# Export Production Artifacts
joblib.dump(xgb_model, "final_model.joblib")
joblib.dump(preprocessor, "preprocessing_pipeline.joblib")
print("✓ Exported final_model.joblib and preprocessing_pipeline.joblib")